# xLSTM Exercise Recognition - Simple Colab Training

Train xLSTM model on EgoExo-Fitness dataset with minimal setup.

## Before You Start
1. Add your HuggingFace token to Colab secrets:
   - Click 🔑 key icon in left sidebar
   - Add secret: `HF_TOKEN = your_token_here`
2. Connect to GPU: Runtime → Change runtime type → GPU → T4

In [ ]:
#@title 1. Install Dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers huggingface_hub pandas tqdm scikit-learn mediapipe opencv-python-headless tensorboard -q

import torch
print(f"✓ PyTorch {torch.__version__}")
print(f"✓ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title 2. Clone Repository & Setup
import os

WORKSPACE = "/content/fitness-coach"
os.chdir(WORKSPACE) if os.path.exists(WORKSPACE) else None

!git clone https://github.com/elinelkonyan/fitness-coach-capstone-1.git {WORKSPACE}
%cd {WORKSPACE}

# Create directories
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

print(f"✓ Working directory: {WORKSPACE}")

In [ ]:
#@title 3. Login to HuggingFace
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('YOUR_HF_TOKEN_HERE')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged in to HuggingFace")
else:
    print("⚠ Add HF_TOKEN to secrets (click 🔑 icon)")
    # Fallback: enter manually
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
#@title 4. Download EgoExo-Fitness Dataset
from huggingface_hub import snapshot_download
import os

DATA_DIR = "/content/data/egoexo"
os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading EgoExo-Fitness dataset...")
print("  This may take 10-30 minutes")

# Download dataset
try:
    snapshot_download(
        repo_id="ego-exo/egoexo-fitness",
        repo_type="dataset",
        local_dir=DATA_DIR,
        token=HF_TOKEN,
        max_workers=4  # Parallel downloads
    )
    print(f"✓ Downloaded to {DATA_DIR}")
    !ls -la {DATA_DIR}
except Exception as e:
    print(f"✗ Download failed: {e}")
    print("  Check that HF_TOKEN is valid and you have access to egoexo-fitness")

In [ ]:
#@title 5. View Dataset Info
import pandas as pd
from pathlib import Path

# Find and load metadata CSV
meta_files = list(Path(DATA_DIR).glob("*.csv"))
print(f"Found metadata files: {[f.name for f in meta_files]}")

if meta_files:
    df = pd.read_csv(meta_files[0])
    print(f"\n✓ Loaded {len(df)} samples")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nExercise classes: {df['exercise_class'].nunique() if 'exercise_class' in df.columns else 'N/A'}")
    print(f"\nSplit distribution:")
    if 'split' in df.columns:
        print(df['split'].value_counts())
    print(f"\nFirst 5 rows:")
    display(df.head())

In [ ]:
#@title 6. Run Training
import subprocess

# Configuration
SMOKE_TEST = True  # Set to False for full training
EPOCHS = 5 if SMOKE_TEST else 100
BATCH_SIZE = 16  # Reduce if OOM
TARGET_FRAMES = 60

# Find the metadata CSV
meta_file = meta_files[0] if meta_files else None

if meta_file:
    cmd = [
        "python", "train_xlstm_exercise.py",
        "--data-csv", str(meta_file),
        "--feature-dir", DATA_DIR,
        "--feature-type", "pose",
        "--target-frames", str(TARGET_FRAMES),
        "--interpolation", "chebyshev",
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--lr", "0.0005",
        "--hidden-size", "128",
        "--num-layers", "2",
        "--output-dir", "/content/results/xlstm",
    ]
    
    print(f"Running training with {EPOCHS} epochs...")
    result = subprocess.run(cmd)
    
    if result.returncode == 0:
        print("\n✓ Training complete!")
    else:
        print("\n✗ Training failed")
else:
    print("⚠ No metadata file found. Check dataset download.")

In [ ]:
#@title 7. View Results
import json
import matplotlib.pyplot as plt

# Load and display results
results_file = "/content/results/xlstm/test_results.json"
history_file = "/content/results/xlstm/training_history.json"

if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    
    print("="*50)
    print("TEST RESULTS")
    print("="*50)
    print(f"Test Accuracy: {results.get('test_accuracy', 0):.4f}")
    
    if 'per_class_metrics' in results:
        print("\nPer-Class Accuracy:")
        for cls, metrics in results['per_class_metrics'].items():
            print(f"  {cls}: {metrics.get('accuracy', 0):.4f} (n={metrics.get('count', 0)})")
else:
    print("⚠ Results not found")

# Plot training history
if os.path.exists(history_file):
    with open(history_file) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'], label='Val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 8. Save to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Create output directory
OUTPUT_DIR = "/content/drive/MyDrive/fitness-coach/xlstm_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy model files
for filename in ['xlstm_best.pt', 'class_map.json', 'test_results.json', 'training_history.json']:
    src = f"/content/results/xlstm/{filename}"
    dst = f"{OUTPUT_DIR}/{filename}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✓ Copied {filename}")
    else:
        print(f"⚠ {filename} not found")

print(f"\n✓ Model saved to: {OUTPUT_DIR}")